# Cell 0 — Git Workflow Guide

## Three-Location Workflow

This notebook is part of a three-location git workflow:
**VS Code (author)** → **GitHub (sync)** → **Colab (compute)** → **GitHub (push back)** → **VS Code (pull)**.

Branch: `qat-wa4-extension`. **Never push to `main` directly.**

---

### Step A — Before opening Colab (run in VS Code terminal)

```bash
git checkout -b qat-wa4-extension
git add training/qat_finetune.py notebooks/qat_colab.ipynb
git commit -m "WA-4 ext: QAT finetune script and Colab notebook"
git push origin qat-wa4-extension
```

### Step B — In Colab

1. Run Cell 2 to clone the repo and check out `qat-wa4-extension`.
2. Run Cells 3–9 in order.
3. Once Cell 8 (export) and Cell 9 (GV1K gate) are both green, run Cell 11 to push artifacts back to GitHub.

### Step C — Back in VS Code

```bash
git pull origin qat-wa4-extension
```

This retrieves `checkpoints/best_model_qat.pth` and `onnx_models/streamsense_multihead.qonnx`.

### Step D — Open PR

Open a pull request from `qat-wa4-extension` into `main` **only after Track E confirms the FINN flow works**. Never merge before Track E sign-off.

---

> **Rule:** Never push directly to `main`. All work lives on `qat-wa4-extension` until Track E confirms.

# Project STREAMSENSE — QAT Fine-tuning and QONNX Export

**Branch:** `qat-wa4-extension`

## What this notebook produces

| Artifact | Path | Description |
|---|---|---|
| `best_model_qat.pth` | `checkpoints/` | QAT fine-tuned checkpoint (Brevitas weights + quantizer scales) |
| `streamsense_multihead.qonnx` | `onnx_models/` | Multi-head QONNX — **Track E FINN flow target** |

The QONNX carries all three output heads:
- `logits`        — `[1, 10]`  float32 — identical classification logits
- `embedding`     — `[1, 128]` float32 — linear projection of the 128-dim GAP vector
- `novelty_score` — `[1, 1]`   float32 — `1 − max(softmax(logits))`, always 2-D

This is the Scope 2 multi-head model in QONNX (Brevitas) format, not the WA-4 QDQ INT8 format. Track E's FINN flow requires QONNX; the WA-4 INT8 QDQ model is not FINN-compatible.

## MPIC v1.0 contract (frozen — do not change)

- Input: `[1, 1, 64, 97]` float32, 16 kHz, 64 mel bins, 97 time frames
- Global norm: mean = −30.785545 dB, std = 22.157099 dB
- Opset 17

In [ ]:
# Cell 2 — Git clone and branch checkout
# REPLACE <YOUR_USERNAME> with your GitHub username before running.

!git clone https://github.com/<YOUR_USERNAME>/STREAMSENSE.git
%cd STREAMSENSE
!git checkout qat-wa4-extension
!git log --oneline -5

In [ ]:
# Cell 3 — Install dependencies
# brevitas pinned to >=0.10 for PyTorch 2.x compatibility on Colab T4.

!pip install -q "brevitas>=0.10" onnx onnxruntime torchaudio

import torch, brevitas, onnx, onnxruntime, torchaudio
print(f"torch        : {torch.__version__}")
print(f"torchaudio   : {torchaudio.__version__}")
print(f"brevitas     : {brevitas.__version__}")
print(f"onnx         : {onnx.__version__}")
print(f"onnxruntime  : {onnxruntime.__version__}")

In [ ]:
# Cell 4 — Mount Google Drive and copy checkpoint
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

src_ckpt = Path('/content/drive/MyDrive/STREAMSENSE_checkpoints/best_model.pth')
dst_ckpt = Path('checkpoints/best_model.pth')
dst_ckpt.parent.mkdir(parents=True, exist_ok=True)

if src_ckpt.exists():
    import shutil
    shutil.copy(str(src_ckpt), str(dst_ckpt))
    size_mb = dst_ckpt.stat().st_size / 1e6
    print(f"[OK] Copied best_model.pth  ({size_mb:.2f} MB) -> {dst_ckpt}")
else:
    print(f"[WARN] Checkpoint not found at: {src_ckpt}")
    print(f"       Please upload best_model.pth manually via the Colab Files panel")
    print(f"       and place it at: {dst_ckpt}")
    print(f"       Or run:  !cp /content/drive/MyDrive/<YOUR_PATH>/best_model.pth {dst_ckpt}")

In [ ]:
# Cell 5 — Verify GPU
!nvidia-smi

import torch
import brevitas

assert torch.cuda.is_available(), "[FAIL] No GPU detected. Change Runtime -> GPU in Colab."
print(f"[PASS] GPU: {torch.cuda.get_device_name(0)}")
print(f"       torch    : {torch.__version__}")
print(f"       brevitas : {brevitas.__version__}")

In [ ]:
# Cell 6 — Download Speech Commands v2
# torchaudio will download and unzip the dataset on first call.

import torchaudio
from pathlib import Path
from collections import Counter

DATA_ROOT = Path('/content/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = {
    "yes": 0, "no": 1, "up": 2, "down": 3,
    "left": 4, "right": 5, "on": 6, "off": 7,
    "stop": 8, "go": 9,
}

# Download validation split (will also download the archive if absent)
print("Downloading / loading Speech Commands v2 (validation split)...")
val_ds = torchaudio.datasets.SPEECHCOMMANDS(root=str(DATA_ROOT), download=True, subset='validation')
print(f"Validation split total clips : {len(val_ds)}")

# Count clips per target class
counter = Counter()
for _, _, label, *_ in val_ds:
    if label in TARGET_CLASSES:
        counter[label] += 1

print("\nClass distribution (validation, target classes only):")
total_target = 0
for label in sorted(TARGET_CLASSES):
    n = counter.get(label, 0)
    total_target += n
    print(f"  {label:<6} : {n}")
print(f"  TOTAL  : {total_target}")

## Cell 7 — QAT Fine-tuning

This cell runs `training/qat_finetune.py` which:

1. Loads `checkpoints/best_model.pth` backbone weights (strict=True).
2. Replaces all `nn.Conv2d` in the backbone with `brevitas.nn.QuantConv2d` (Int8 per-tensor).
3. Replaces all `nn.Linear` in backbone classifier and `embed_head` with `brevitas.nn.QuantLinear`.
4. Applies the Brevitas device-placement fix (`model.to(device)` last, then buffer verification).
5. **Epochs 1–3**: backbone frozen — only `embed_head` weights and Brevitas quantizer scale factors are trained.
6. **Epoch 4+**: all parameters unfrozen for full QAT fine-tuning.
7. Saves best checkpoint by validation top-1 accuracy.
8. Runs the GV1K gate after training — hard exit if top-1 < 90 %.

Expected runtime on T4: ~20–40 minutes for 10 epochs.

In [ ]:
# Cell 7 — Run QAT fine-tuning
!python training/qat_finetune.py \
    --ckpt checkpoints/best_model.pth \
    --data /content/data \
    --epochs 10 \
    --lr 1e-5 \
    --out checkpoints/best_model_qat.pth \
    --gvk golden_vectors_1000/normalized \
    --device cuda

In [ ]:
# Cell 8 — Export QONNX (full inline implementation)
#
# This cell does NOT import from qat_finetune.py.  All Brevitas replacements
# are written out in full here so the export is self-contained and reproducible
# without re-running training.

import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

import onnx
import onnxruntime as ort

import brevitas.nn as qnn
from brevitas.quant import Int8WeightPerTensorFloat, Int8ActPerTensorFloat
from brevitas.export import export_qonnx

# ── Project paths ─────────────────────────────────────────────────────────────
ROOT = Path('/content/STREAMSENSE')
sys.path.insert(0, str(ROOT / 'training'))

from model import StreamSenseNet
from streaming_wrapper import StreamSenseWrapper, NUM_CLASSES, EMBEDDING_DIM

CKPT_PATH   = ROOT / 'checkpoints' / 'best_model_qat.pth'
ORIG_CKPT   = ROOT / 'checkpoints' / 'best_model.pth'
OUT_DIR     = ROOT / 'onnx_models'
EXPORT_PATH = OUT_DIR / 'streamsense_multihead.qonnx'

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Brevitas module replacement functions (identical to qat_finetune.py) ──────

def _replace_conv2d(module):
    for name, child in module.named_children():
        if isinstance(child, nn.Conv2d):
            qconv = qnn.QuantConv2d(
                in_channels  = child.in_channels,
                out_channels = child.out_channels,
                kernel_size  = child.kernel_size,
                stride       = child.stride,
                padding      = child.padding,
                dilation     = child.dilation,
                groups       = child.groups,
                bias         = child.bias is not None,
                weight_quant = Int8WeightPerTensorFloat,
                input_quant  = Int8ActPerTensorFloat,
                output_quant = Int8ActPerTensorFloat,
                return_quant_tensor = False,
            )
            with torch.no_grad():
                qconv.weight.copy_(child.weight)
                if child.bias is not None and qconv.bias is not None:
                    qconv.bias.copy_(child.bias)
            setattr(module, name, qconv)
        else:
            _replace_conv2d(child)
    return module


def _replace_linear(module):
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            qlin = qnn.QuantLinear(
                in_features  = child.in_features,
                out_features = child.out_features,
                bias         = child.bias is not None,
                weight_quant = Int8WeightPerTensorFloat,
                input_quant  = Int8ActPerTensorFloat,
                output_quant = Int8ActPerTensorFloat,
                return_quant_tensor = False,
            )
            with torch.no_grad():
                qlin.weight.copy_(child.weight)
                if child.bias is not None and qlin.bias is not None:
                    qlin.bias.copy_(child.bias)
            setattr(module, name, qlin)
        else:
            _replace_linear(child)
    return module


# ── Build model structure (CPU — export always runs on CPU) ───────────────────
device = torch.device('cpu')
model  = StreamSenseWrapper(num_classes=NUM_CLASSES)

# Load original backbone weights to initialise the structure before applying
# QAT replacements (required so shape metadata is correct when replacing layers).
orig_ckpt = torch.load(ORIG_CKPT, map_location='cpu', weights_only=True)
model.backbone.load_state_dict(orig_ckpt['model_state'], strict=True)
print(f"[setup] Loaded backbone from epoch {orig_ckpt.get('epoch','?')}")

# Apply Brevitas replacements
_replace_conv2d(model.backbone.block1)
_replace_conv2d(model.backbone.block2)
_replace_conv2d(model.backbone.block3)
_replace_linear(model.backbone.classifier)
_replace_linear(model.embed_head)

# Brevitas device-placement fix: model.to(device) LAST
model.to(device)

# Mandatory buffer verification
for buf_name, buf in model.named_buffers():
    assert buf.device.type == device.type, (
        f"[device-check] Buffer {buf_name!r} is on {buf.device.type!r}, "
        f"expected {device.type!r}. Brevitas device-placement bug."
    )
print("[device-check] All buffers on CPU — OK")

# Load QAT checkpoint weights (overrides all layers including quantizer scales)
qat_ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=True)
model.load_state_dict(qat_ckpt['model_state'])
print(f"[setup] Loaded QAT checkpoint  epoch={qat_ckpt.get('epoch','?')}  "
      f"val_acc={qat_ckpt.get('val_accuracy', float('nan')):.2f}%")

model.eval()

# ── Export QONNX ──────────────────────────────────────────────────────────────
input_t = torch.zeros(1, 1, 64, 97, dtype=torch.float32)

print(f"\n[export] Exporting QONNX -> {EXPORT_PATH}")
export_qonnx(
    module      = model,
    input_t     = input_t,
    export_path = str(EXPORT_PATH),
)
print(f"[export] Done.")

# ── ONNX structural check ──────────────────────────────────────────────────────
qonnx_model = onnx.load(str(EXPORT_PATH))
onnx.checker.check_model(qonnx_model)
print("[onnx.checker] PASS — model is structurally valid")

# ── ORT sanity inference ───────────────────────────────────────────────────────
# Use qonnx-compatible runtime (onnxruntime with qonnx extensions if needed)
sess = ort.InferenceSession(str(EXPORT_PATH), providers=['CPUExecutionProvider'])

dummy_np = np.zeros((1, 1, 64, 97), dtype=np.float32)
outputs  = sess.run(None, {sess.get_inputs()[0].name: dummy_np})
out_names = [o.name for o in sess.get_outputs()]

print("\n[ORT sanity] Output names and shapes:")
for name, arr in zip(out_names, outputs):
    print(f"  {name:<20} : {tuple(arr.shape)}")

# Build a dict for assertion checks
out_dict = {name: arr for name, arr in zip(out_names, outputs)}

# ── Shape assertions ──────────────────────────────────────────────────────────
logits_shape  = tuple(out_dict['logits'].shape)
embed_shape   = tuple(out_dict['embedding'].shape)
novelty_shape = tuple(out_dict['novelty_score'].shape)

logits_ok  = (logits_shape  == (1, 10))
embed_ok   = (embed_shape   == (1, 128))
novelty_ok = (novelty_shape == (1, 1))

print(f"\n[shape assert] logits        {logits_shape}   : {'PASS' if logits_ok  else 'FAIL ← EXPECTED (1, 10)'}")
print(f"[shape assert] embedding     {embed_shape}  : {'PASS' if embed_ok   else 'FAIL ← EXPECTED (1, 128)'}")
print(f"[shape assert] novelty_score {novelty_shape}    : {'PASS' if novelty_ok else 'FAIL ← EXPECTED (1, 1) — check keepdim=True in forward()'}")

assert logits_ok,  f"logits shape {logits_shape} != (1, 10)  — ERR v1.0 contract broken"
assert embed_ok,   f"embedding shape {embed_shape} != (1, 128) — ERR v1.0 contract broken"
assert novelty_ok, f"novelty_score shape {novelty_shape} != (1, 1)  — ERR v1.0 contract broken"

print("\n[PASS] All three output shapes conform to ERR v1.0 contract.")

# ── File size ─────────────────────────────────────────────────────────────────
size_mb = EXPORT_PATH.stat().st_size / 1e6
print(f"[export] File size: {size_mb:.2f} MB  -> {EXPORT_PATH}")

In [ ]:
# Cell 9 — GV1K top-1 verification on exported QONNX

import sys
import numpy as np
import onnxruntime as ort
from pathlib import Path

ROOT        = Path('/content/STREAMSENSE')
QONNX_PATH  = ROOT / 'onnx_models' / 'streamsense_multihead.qonnx'
GV1K_DIR    = ROOT / 'golden_vectors_1000' / 'normalized'

TARGET_CLASSES = {
    "yes": 0, "no": 1, "up": 2, "down": 3,
    "left": 4, "right": 5, "on": 6, "off": 7,
    "stop": 8, "go": 9,
}


def _parse_label(stem: str):
    """
    Parse ground-truth class index from GV1K filename stem.
    Pattern: GV1K_NNNN_<label>_norm
    Returns int index or None if unrecognised.
    """
    parts = stem.split('_')
    if len(parts) < 4:
        return None
    return TARGET_CLASSES.get(parts[2].lower(), None)


# ── Load ONNX session ─────────────────────────────────────────────────────────
sess      = ort.InferenceSession(str(QONNX_PATH), providers=['CPUExecutionProvider'])
in_name   = sess.get_inputs()[0].name
out_names = [o.name for o in sess.get_outputs()]
print(f"QONNX input: {in_name!r}")
print(f"Outputs    : {out_names}")

# ── Load GV1K vectors ─────────────────────────────────────────────────────────
bin_files = sorted(GV1K_DIR.glob('*_norm.bin'))
print(f"\nGV1K vectors found : {len(bin_files)}")

if len(bin_files) == 0:
    print(f"[WARN] No *_norm.bin files found in {GV1K_DIR}")
    print("       Generate GV1K vectors first: python training/generate_golden_1000.py")

# ── Run inference on all 1000 vectors ────────────────────────────────────────
correct = 0
wrong   = 0
skipped = 0

for bf in bin_files:
    true_idx = _parse_label(bf.stem)
    if true_idx is None:
        skipped += 1
        continue

    raw = np.fromfile(str(bf), dtype='<f4')
    if raw.size != 64 * 97:
        skipped += 1
        continue

    inp    = raw.reshape(1, 1, 64, 97).astype(np.float32)
    outs   = sess.run(None, {in_name: inp})
    out_d  = {name: arr for name, arr in zip(out_names, outs)}
    logits = out_d['logits']                   # [1, 10]
    pred   = int(np.argmax(logits[0]))

    if pred == true_idx:
        correct += 1
    else:
        wrong += 1

total_checked = correct + wrong
top1_acc      = 100.0 * correct / total_checked if total_checked > 0 else 0.0

print(f"\n{'='*50}")
print(f"GV1K QONNX Verification")
print(f"{'='*50}")
print(f"  Total checked  : {total_checked}")
print(f"  Correct        : {correct}")
print(f"  Wrong          : {wrong}")
print(f"  Skipped        : {skipped}")
print(f"  Top-1 accuracy : {top1_acc:.2f}%")

gate_passed = (top1_acc >= 90.0)
print(f"\n[{'PASS' if gate_passed else 'FAIL'}] GV1K gate {'passed' if gate_passed else 'FAILED'} "
      f"({top1_acc:.2f}% {'≥' if gate_passed else '<'} 90.0%)")

assert gate_passed, (
    f"GV1K gate FAILED: {top1_acc:.2f}% < 90.0% minimum.  "
    f"Do NOT push — debug the QONNX export first."
)

## Cell 10 — Review before pushing

**Before running Cell 11**, confirm the following in the Cell 8 and Cell 9 outputs:

- `[PASS] All three output shapes conform to ERR v1.0 contract.`
  - `logits` : `(1, 10)`
  - `embedding` : `(1, 128)`
  - `novelty_score` : `(1, 1)` — must be exactly 2-D
- `[PASS] GV1K gate passed (≥ 90.0%)`
- `[onnx.checker] PASS`

If **any** of these is red or shows FAIL, **do not run Cell 11**. Debug the export first:
- If `novelty_score` is `(1,)` (1-D), check that `keepdim=True` is present in `StreamSenseWrapper.forward()`.
- If GV1K accuracy is < 90 %, re-run training with more epochs or a slightly higher learning rate.
- If the ONNX checker fails, check that the Brevitas version is compatible with the installed PyTorch.

In [ ]:
# Cell 11 — Git commit and push artifacts back to GitHub
#
# REPLACE the email, name, and token/username before running.
# If push fails with authentication error, set the remote URL with your PAT:
#   !git remote set-url origin https://<TOKEN>@github.com/<USERNAME>/STREAMSENSE.git

!git config user.email "your.email@example.com"   # REPLACE with your email
!git config user.name  "Your Name"                 # REPLACE with your name

!git add checkpoints/best_model_qat.pth
!git add onnx_models/streamsense_multihead.qonnx
!git add notebooks/qat_colab.ipynb

!git commit -m "WA-4 ext: QAT checkpoint and QONNX multihead export — 3-output Track E target — GV1K green"
!git push origin qat-wa4-extension

print()
print("If the push failed due to authentication, run this in a code cell:")
print("  !git remote set-url origin https://<TOKEN>@github.com/<USERNAME>/STREAMSENSE.git")
print("Then re-run this cell.")

In [ ]:
# Cell 12 — Copy artifacts to Google Drive for local backup
import shutil
from pathlib import Path

drive_out = Path('/content/drive/MyDrive/STREAMSENSE_outputs')
drive_out.mkdir(parents=True, exist_ok=True)

src_ckpt  = Path('checkpoints/best_model_qat.pth')
src_qonnx = Path('onnx_models/streamsense_multihead.qonnx')
dst_ckpt  = drive_out / 'best_model_qat.pth'
dst_qonnx = drive_out / 'streamsense_multihead.qonnx'

shutil.copy(str(src_ckpt),  str(dst_ckpt))
shutil.copy(str(src_qonnx), str(dst_qonnx))

print(f"[copy] best_model_qat.pth          : {dst_ckpt.stat().st_size / 1e6:.2f} MB  -> {dst_ckpt}")
print(f"[copy] streamsense_multihead.qonnx : {dst_qonnx.stat().st_size / 1e6:.2f} MB  -> {dst_qonnx}")

## Cell 13 — Local VS Code pull instructions

After Colab has pushed successfully, run the following in the VS Code terminal:

```bash
git pull origin qat-wa4-extension
```

Confirm both new files appear locally:
- `checkpoints/best_model_qat.pth`
- `onnx_models/streamsense_multihead.qonnx`

### Local QONNX sanity check (optional but recommended)

Run a quick ORT inference on CPU locally to confirm the file is intact:

```python
import numpy as np
import onnxruntime as ort
sess = ort.InferenceSession('onnx_models/streamsense_multihead.qonnx',
                             providers=['CPUExecutionProvider'])
out  = sess.run(None, {'input': np.zeros((1,1,64,97), dtype='float32')})
for o, arr in zip(sess.get_outputs(), out):
    print(o.name, arr.shape)
```

Expected output:
```
logits        (1, 10)
embedding     (1, 128)
novelty_score (1, 1)
```

### Hand-off to Track E

Give Track E these two files:
- `onnx_models/streamsense_multihead.qonnx` — **FINN flow target** (QONNX format, all 3 heads)
- `onnx_models/streamsense_multihead_fp32.onnx` — FP32 reference for ORT inference

### Open PR

Open a pull request from `qat-wa4-extension` into `main` **only after Track E confirms the FINN flow works**.
Never merge before Track E sign-off.

## Cell 14 — Handover Summary

| File | Path | For whom | Purpose |
|---|---|---|---|
| `streamsense_multihead.qonnx` | `onnx_models/` | Track E | FINN flow target — all 3 heads, QONNX format (Brevitas QAT) |
| `streamsense_multihead_fp32.onnx` | `onnx_models/` | Track E | FP32 reference — ORT inference, all 3 heads |
| `best_model_qat.pth` | `checkpoints/` | Internal Track A | QAT checkpoint — source of QONNX, Brevitas quantizer scales included |
| `streamsense_multihead_int8.onnx` | `onnx_models/` | Internal Track A | QDQ PTQ reference — 95.8% GV1K, NOT FINN-compatible |